# MLP on MNIST — Figure 2(a) Reproduction

Reproduction of Figure 2(a) from *Adam: A Method for Stochastic Optimization* (Kingma & Ba, 2015).

**Experiment**: Two-layer fully connected neural network on MNIST with dropout regularisation.

| Parameter | Value | Source |
|---|---|---|
| Dataset | MNIST (60 k train / 10 k test) | paper |
| Architecture | 2 hidden layers × 1000 units, ReLU | paper |
| Minibatch size | 128 | paper |
| Dropout rate | 0.5 | *(our choice)* |
| L2 weight decay λ | None (for stochastic) | paper |
| Weight initialisation | Glorot uniform (Xavier) | *(our choice)* |
| Adam β₁, β₂, ε | 0.9, 0.999, 10⁻⁸ | paper (default) |
| SGD momentum | 0.9 | *(our choice)* |
| All optimisers α | grid search (see below) | paper: "dense grid, best reported" |
| Training epochs | 200 | inferred from Fig. 2(a) x-axis |

### Notes

Unlike Section 6.1, the paper does **not** apply a 1/√t learning-rate schedule here.
Fixed base learning rates are used, selected via grid search.

**Figure 2(b)** in the original paper compares Adam against the Sum-of-Functions optimizer
(SFO, Sohl-Dickstein et al., 2014) on a deterministic objective (no dropout).
SFO is not re-implemented in this study. A proxy comparison of Adam vs. SGDNesterov on
the deterministic objective is provided in the final section.

**Training cost** = average negative log-likelihood per sample on the current mini-batch,
smoothed over a window of one epoch (= BATCHES_PER_EPOCH, set after data loading).

In [ ]:
import os
import copy

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Configuration
NUM_EPOCHS   = 200
BATCH_SIZE   = 128
DROPOUT_P    = 0.5
WEIGHT_DECAY = 1e-4

# Per-optimizer LR grids (paper: "dense grid, best hyper-parameter setting reported")
# No 1/sqrt(t) schedule here — fixed LR throughout (unlike Sec. 6.1).
LR_GRID_ADAM     = [1e-4, 1e-3, 1e-2]
LR_GRID_SGD      = [1e-2, 1e-1, 3e-1, 1.0]
LR_GRID_ADAGRAD  = [1e-3, 1e-2, 1e-1, 3e-1]
LR_GRID_RMSPROP  = [1e-4, 1e-3, 1e-2]
LR_GRID_ADADELTA = [1e-1, 1.0, 5.0]   # lr is a global scaling factor for AdaDelta

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_dataset = datasets.MNIST(root='data/', train=True, download=False, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

BATCHES_PER_EPOCH = len(train_loader)
SMOOTH_WINDOW     = BATCHES_PER_EPOCH   # smooth over exactly one epoch
print(f'Samples: {len(train_dataset)},  Batches/epoch: {BATCHES_PER_EPOCH}')

In [ ]:
# Model
class MLP(nn.Module):
    # Two hidden layers x 1000 ReLU units; optional Dropout after each hidden layer.
    def __init__(self, dropout_p=0.5):
        super().__init__()
        layers = [nn.Linear(784, 1000), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        layers += [nn.Linear(1000, 1000), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        layers.append(nn.Linear(1000, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_model(dropout_p=DROPOUT_P):
    # Factory: create, apply Glorot uniform init, move to DEVICE.
    model = MLP(dropout_p=dropout_p).to(DEVICE)
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


# Training loop 
def train(model, optimizer, num_epochs=NUM_EPOCHS):
    # Train model, return per-batch NLL list.
    criterion = nn.CrossEntropyLoss()
    batch_losses = []
    model.train()
    for epoch in range(num_epochs):
        for images, labels in train_loader:
            images = images.view(images.size(0), -1).to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        if (epoch + 1) % 20 == 0:
            avg = np.mean(batch_losses[-BATCHES_PER_EPOCH:])
            print(f'  Epoch {epoch+1:3d}/{num_epochs}  avg NLL={avg:.4f}')
    return batch_losses


def smooth(losses, window=None):
    # Running-average smoothing (mode='valid').
    w = window if window is not None else SMOOTH_WINDOW
    return np.convolve(losses, np.ones(w) / w, mode='valid')


# Grid search 
def grid_search(optimizer_class, optimizer_kwargs, lr_grid, dropout_p=DROPOUT_P):
    # Run one training per LR candidate; return (best_lr, best_losses, all_results).
    best_lr, best_metric, best_losses = None, float('inf'), []
    all_results = {}
    for lr in lr_grid:
        print(f'\n  lr = {lr}')
        model = make_model(dropout_p=dropout_p)
        kw = copy.deepcopy(optimizer_kwargs)
        kw['lr'] = lr
        opt = optimizer_class(model.parameters(), **kw)
        losses = train(model, opt)
        smoothed = smooth(losses)
        tail = smoothed[-10 * BATCHES_PER_EPOCH:]
        metric = float(np.mean(tail))
        print(f'  tail-mean NLL = {metric:.4f}')
        all_results[lr] = losses
        if metric < best_metric:
            best_metric, best_lr, best_losses = metric, lr, losses
    print(f'\n  Best lr = {best_lr}  (tail-mean NLL = {best_metric:.4f})')
    return best_lr, best_losses, all_results


# olour palette
COLORS = {
    'Adam':        'tab:purple',
    'SGDNesterov': 'tab:red',
    'AdaGrad':     'tab:blue',
    'RMSProp':     'tab:green',
    'AdaDelta':    'tab:cyan',
}

results_stoch = {}  

---
## Figure 2(a) — Stochastic objective (with dropout)

All five optimisers are trained on the MLP with Dropout (p = 0.5).
No weight decay or learning-rate schedule is applied — each run uses a constant α selected by grid search.

### 1. Adam

Default hyperparameters β₁ = 0.9, β₂ = 0.999, ε = 10⁻⁸ (from paper).
Grid search over α ∈ {10⁻⁴, 10⁻³, 10⁻²}.

In [ ]:
print('Grid search: Adam (stochastic)')
best_lr, best_losses, adam_all = grid_search(
    torch.optim.Adam,
    {'weight_decay': WEIGHT_DECAY, 'betas': (0.9, 0.999), 'eps': 1e-8},
    LR_GRID_ADAM,
)

results_stoch['Adam'] = {'lr': best_lr, 'losses': best_losses}

### 2. SGD + Nesterov momentum

Momentum = 0.9 (*our choice*).
Grid search over α ∈ {10⁻², 10⁻¹, 0.3, 1.0}.

**Expected behaviour**: without per-parameter learning-rate adaptation, SGD with momentum
accumulates noisy gradients from dropout without suppressing the noise per dimension,
making it the weakest method in this setting.

In [ ]:
print('Grid search: SGDNesterov (stochastic)')
best_lr, best_losses, sgd_all = grid_search(
    torch.optim.SGD,
    {'weight_decay': WEIGHT_DECAY, 'momentum': 0.9, 'nesterov': True},
    LR_GRID_SGD,
)

results_stoch['SGDNesterov'] = {'lr': best_lr, 'losses': best_losses}

### 3. AdaGrad

AdaGrad's monotonically growing squared-gradient accumulator is less problematic here than
in Section 6.1 (no compounding 1/√t schedule), but training still tends to stall after
~50–100 epochs as per-parameter learning rates shrink.
Grid search over α ∈ {10⁻³, 10⁻², 10⁻¹, 0.3}.

In [ ]:
print('Grid search: AdaGrad (stochastic)')
best_lr, best_losses, adagrad_all = grid_search(
    torch.optim.Adagrad,
    {'weight_decay': WEIGHT_DECAY},
    LR_GRID_ADAGRAD,
)

results_stoch['AdaGrad'] = {'lr': best_lr, 'losses': best_losses}

### 4. RMSProp

RMSProp (Tieleman & Hinton, 2012): exponential moving average of squared gradients,
no bias correction, no first-moment tracking. PyTorch `alpha` = β₂ decay rate = 0.99.
Grid search over α ∈ {10⁻⁴, 10⁻³, 10⁻²}.

In [ ]:
print('Grid search: RMSProp (stochastic)')
best_lr, best_losses, rmsprop_all = grid_search(
    torch.optim.RMSprop,
    {'weight_decay': WEIGHT_DECAY, 'alpha': 0.99, 'eps': 1e-8},
    LR_GRID_RMSPROP,
)

results_stoch['RMSProp'] = {'lr': best_lr, 'losses': best_losses}

### 5. AdaDelta

AdaDelta (Zeiler, 2012) requires no explicit learning rate in its original formulation.
PyTorch's `lr` parameter is a global scaling factor (default 1.0); `rho` = 0.9 (default).
Grid search over the scaling factor ∈ {0.1, 1.0, 5.0}.

In [ ]:
print('Grid search: AdaDelta (stochastic)')
best_lr, best_losses, adadelta_all = grid_search(
    torch.optim.Adadelta,
    {'weight_decay': WEIGHT_DECAY, 'rho': 0.9, 'eps': 1e-6},
    LR_GRID_ADADELTA,
)

results_stoch['AdaDelta'] = {'lr': best_lr, 'losses': best_losses}

## Plot — Figure 2(a)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for name in ['Adam', 'SGDNesterov', 'AdaGrad', 'RMSProp', 'AdaDelta']:
    data = results_stoch[name]
    smoothed = smooth(data['losses'])
    x = np.arange(len(smoothed)) / BATCHES_PER_EPOCH
    ax.semilogy(x, smoothed, color=COLORS[name], linewidth=1.5,
                label=f"{name} (\u03b1={data['lr']})")

ax.set_title('MLP + Dropout Training Cost (MNIST)', fontsize=13)
ax.set_xlabel('Epochs')
ax.set_ylabel('Training Cost (NLL, log scale)')
ax.set_xlim(0, NUM_EPOCHS)
ax.set_ylim(1e-3, 1e0)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

fig.text(
    0.5, -0.02,
    'Reproduction of Figure 2(a) in Kingma & Ba (2015). '
    'Training cost of a two-layer MLP on MNIST with dropout regularisation.',
    ha='center', fontsize=9, style='italic',
)

os.makedirs('results', exist_ok=True)
fig.savefig('results/fig2a_reproduction.png', dpi=250, bbox_inches='tight')
plt.show()
print('Saved to results/fig2a_reproduction.png')